# Exploratory Analysis

This notebook provides a quick review of the sample food price dataset used by the Streamlit early warning app.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import preprocess_price_data, filter_data
from src.indicators import add_market_indicators
from src.anomaly_detection import classify_alerts


## Load Data

In [ ]:
from src.data_loader import load_price_data

raw = load_price_data()
raw.head()

In [ ]:
raw.info()
raw.describe(include="all")

## Prepare and Inspect Coverage

In [ ]:
data = preprocess_price_data(raw)
coverage = data.groupby(["country", "market", "commodity"]).agg(
    observations=("price", "size"),
    first_date=("date", "min"),
    last_date=("date", "max"),
    mean_price=("price", "mean"),
).reset_index()
coverage.head(10)

## Indicator Example

In [ ]:
example_row = data.iloc[0]
example = filter_data(
    data=data,
    country=example_row["country"],
    market=example_row["market"],
    commodity=example_row["commodity"],
    start_date=data["date"].min().date(),
    end_date=data["date"].max().date(),
)

series = classify_alerts(add_market_indicators(example))
series.tail()

In [ ]:
series["alert_level"].value_counts()

In [ ]:
fig = px.line(
    series,
    x="date",
    y=["price", "rolling_mean_6m"],
    title=f"{example_row['market']} {example_row['commodity']} Price Index and 6-Month Rolling Mean",
)
fig.show()